In [1]:
import argparse
import logging
import os
import torch
import numpy as np
import random
import json
# To set deterministic behaviour:
os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'  # or ':16:8'

from mmengine.config import Config, DictAction
from mmengine.logging import print_log
from mmengine.registry import RUNNERS
from mmengine.runner import Runner
from mmdet.evaluation import DumpDetResults

from mmdet.utils import setup_cache_size_limit_of_dynamo



base_folder = '/Data_large/marine/PythonProjects/MMDET/MyConfigs'
cfg = Config.fromfile(f'{base_folder}/Venus_b5/vfnet_r18.py')

setup_cache_size_limit_of_dynamo()

def set_seed(seed):
    # Set the seed for generating random numbers in PyTorch
    torch.manual_seed(seed)
    # If using GPUs, ensure that the random numbers are generated the same way
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)  # if you are using multi-GPU.
    
    # Set the seed for generating random numbers in Python
    random.seed(seed)
    
    # Set the seed for generating random numbers in numpy
    np.random.seed(seed)
    
    # Ensure deterministic behavior by setting the flag
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    
    # Optionally, set environment variables to ensure reproducibility
    os.environ['PYTHONHASHSEED'] = str(seed)

### Custom

In [2]:
BAND_SEL = 5 # Selecting the Band for Venus

In [3]:
AMP = False
# Normalization:
MEAN_VALS = [88.4]
STD_VALS = [64.5]
# Resizing:
IMG_SIZE = 2048

# Training:
BS = 8
LR = 0.001

## Testing:
ann_file = '/Data_large/marine/Datasets/VENuS/annotations/perfect/test.json'
data_root = '/Data_large/marine/Datasets/VENuS/ds_L0/'
data_prefix = f'perfect_b{BAND_SEL}/'

# Deterministic Behaviour setting:
SEED = 41
set_seed(SEED)
cfg.randomness = dict(
    seed = SEED, # 41 72 18
    diff_rank_seed=True,
    # deterministic=True
)

MAX_EPOCHS = 20

optimizers =  {'SGD':{'type': 'OptimWrapper', 'optimizer': {'type': 'SGD', 'lr': LR, 'momentum': 0.9, 'weight_decay': 0.0001}},
            'Adam':{'type': 'OptimWrapper', 'optimizer': {'type': 'Adam', 'lr': LR, 'weight_decay': 0.0001}},
            'AdamW':{'type': 'OptimWrapper', 'optimizer': {'type': 'Adam', 'lr': LR, 'weight_decay': 0.0001}},}
selOpt = 'SGD'

# Savedir
workdir = f'/Data_large/marine/PythonProjects/MMDET/checkpoints/VENuS/perfect_b{BAND_SEL}/{SEED}_BS_{BS}_LR_{LR}_ME_{MAX_EPOCHS}_OPT_{selOpt}'

#### Config 

In [ ]:
#### WORKDIR
cfg.work_dir = workdir

#### AMP
# enable automatic-mixed-precision training
if AMP is True:
    optim_wrapper = cfg.optim_wrapper.type
    if optim_wrapper == 'AmpOptimWrapper':
        print_log(
            'AMP training is already enabled in your config.',
            logger='current',
            level=logging.WARNING)
    else:
        assert optim_wrapper == 'OptimWrapper', (
            '`--amp` is only supported when the optimizer wrapper type is '
            f'`OptimWrapper` but got {optim_wrapper}.')
        cfg.optim_wrapper.type = 'AmpOptimWrapper'
        cfg.optim_wrapper.loss_scale = 'dynamic'

# Dataloader:
cfg.model.data_preprocessor.mean = [float(x) for x in MEAN_VALS]
cfg.model.data_preprocessor.std = [float(x) for x in STD_VALS]

cfg.train_dataloader.dataset.data_prefix = {'img':f'ds_L0/perfect_b{BAND_SEL}/'}
cfg.train_dataloader.dataset.pipeline[3] = {'type': 'Resize', 'scale': (IMG_SIZE, IMG_SIZE), 'keep_ratio': False}
cfg.val_dataloader.dataset.pipeline[2] = {'type': 'Resize', 'scale': (IMG_SIZE, IMG_SIZE), 'keep_ratio': False}


# Training params:
cfg.train_dataloader.batch_size = BS

cfg.train_cfg = {'type': 'EpochBasedTrainLoop', 'max_epochs': MAX_EPOCHS, 'val_interval': 1}

# TODO: implement stages as in: https://github.com/open-mmlab/mmdetection/blob/cfd5d3a985b0249de009b67d04f37263e11cdf3d/configs/rtmdet/rtmdet_x_p6_4xb8-300e_coco.py#L78
# lr_config = dict(policy='poly', power=0.9, min_lr=1e-4, by_epoch=False)

cfg.optim_wrapper = optimizers[selOpt]

cfg.param_scheduler = [{'type': 'LinearLR',
                        'start_factor': 0.001,
                        'by_epoch': True,
                        'begin': 0,
                        'end': 5},
                        {'type': 'MultiStepLR',
                        'begin': 0,
                        'end': MAX_EPOCHS//2,
                        'by_epoch': True,
                        'milestones': [MAX_EPOCHS//4, MAX_EPOCHS//3, MAX_EPOCHS//2],
                        'gamma': 0.75}, 
                        {# use cosine lr from 150 to 300 epoch
                        'type':'CosineAnnealingLR',
                        'eta_min':LR * 0.05,
                        'begin':MAX_EPOCHS // 2,
                        'end':MAX_EPOCHS,
                        'T_max':MAX_EPOCHS // 2,
                        'by_epoch':True,
                        'convert_to_iter_based':True,}
                        ]

#### Test Config hooks:
default_hooks = cfg.default_hooks
if 'visualization' in default_hooks:
    visualization_hook = default_hooks['visualization']
    # Turn on visualization
    visualization_hook['draw'] = False

cfg.test_dataloader = dict(
            batch_size=1,
            dataset=dict(
                ann_file=ann_file,
                data_root=data_root,
                data_prefix=dict(img=data_prefix),
                filter_cfg=dict(filter_empty_gt=True),
                metainfo=dict(classes=('Vessel', ), palette=[
                    (
                        220,
                        20,
                        60,
                    ),
                ]),
                pipeline=[
                    dict(
                        backend_args=None,
                        color_type='color',
                        imdecode_backend='tifffile',
                        to_float32=True,
                        type='LoadImageFromFile'),
                    dict(type='LoadAnnotations', with_bbox=True),
                    dict(keep_ratio=False, scale=(IMG_SIZE,IMG_SIZE,), type='Resize'),
                    dict(
                        meta_keys=('img_path', 'img_id', 'seg_map_path', 
                                'height', 'width', 'instances', 'sample_idx', 
                                'img', 'img_shape', 'ori_shape', 'scale', 'scale_factor', 
                                'keep_ratio', 'homography_matrix', 'gt_bboxes', 'gt_ignore_flags', 
                                'gt_bboxes_labels'),
                        type='PackDetInputs'),
                ],
                test_mode=True,
                type='CocoDataset'),
            drop_last=False,
            num_workers=2,
            persistent_workers=True,
            sampler=dict(shuffle=False, type='DefaultSampler'))

cfg.test_evaluator = dict(
            type='CocoMetric',
            metric='bbox',
            format_only=False,
            ann_file=ann_file,
            outfile_prefix=f'{workdir}/test_results')


# build the runner from config
if 'runner_type' not in cfg:
    # build the default runner
    runner = Runner.from_cfg(cfg)
else:
    # build customized runner from the registry
    # if 'runner_type' is set in the cfg
    runner = RUNNERS.build(cfg)

In [ ]:
runner.train()

In [ ]:
runner.test_evaluator.metrics.append(DumpDetResults(out_file_path=f'{workdir}/test_result/test.pkl'))
# start testing
output_test_data =runner.test()

# Specify the file name
file_name = f'{workdir}/test_result/coco_metrics.json'# Specify the filepath
# Write the dictionary to a JSON file
with open(file_name, 'w') as json_file:
    json.dump(output_test_data, json_file, indent=4)

print(f"Data has been saved to {file_name}")